# Dynamic Bayesian networks — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Repeat a two-slice template through time
Dynamic Bayesian networks generalize HMMs by tying a local temporal graph across slices. These examples unroll a two-state template, filter with evidence, compare tied and untied parameters, and show parameter savings.

In [ ]:
# 1) unroll a two-slice transition template
pi=np.array([0.7,0.3]); T=np.array([[0.8,0.2],[0.25,0.75]]); b=pi
hist=[b]
for _ in range(4): b=b@T; hist.append(b)
plt.figure(figsize=(6,3)); plt.plot(np.array(hist)); plt.legend(['state0','state1']); plt.title('repeated transition template')
assert abs(hist[-1].sum()-1)<1e-12

In [ ]:
# 2) add a sensor model for filtering
E=np.array([[0.9,0.1],[0.3,0.7]]); obs=[0,0,1,1]; b=pi; vals=[]
for o in obs: b=(b@T)*E[:,o]; b=b/b.sum(); vals.append(b[1])
plt.figure(figsize=(6,3)); plt.plot(vals,'-o'); plt.title('evidence updates each slice')
assert vals[-1]>vals[0]

In [ ]:
# 3) prediction without evidence drifts toward stationary behavior
b=pi; preds=[]
for _ in range(10): b=b@T; preds.append(b[1])
plt.figure(figsize=(6,3)); plt.plot(preds,'-o'); plt.title('multi-step DBN prediction')
assert preds[-1]>0.4 and preds[-1]<0.5

In [ ]:
# 4) tied parameters save tables across time
Tlen=10; untied=(Tlen-1)*2; tied=2
plt.figure(figsize=(6,3)); plt.bar(['untied transitions','tied template'],[untied,tied]); plt.title('template sharing reduces parameters')
assert untied==18 and tied==2

In [ ]:
# 5) changing the template changes long-run belief
T2=np.array([[0.95,0.05],[0.1,0.9]]); b1=pi; b2=pi
for _ in range(20): b1=b1@T; b2=b2@T2
plt.figure(figsize=(6,3)); plt.bar(['original P(state1)','sticky P(state1)'],[b1[1],b2[1]]); plt.title('template encodes dynamics')
assert b2[0]>b1[0]